# NB03 — Joblib and Dask for Parallel Biology

**Module 17: HPC, Parallel Computing, and Rust**

---

## 1. Motivation

`multiprocessing.Pool` is low-level: you manage worker pools, handle pickling, and write boilerplate. **Joblib** raises the abstraction — one decorator-like call parallelizes a comprehension, with automatic memory caching for expensive intermediate results. **Dask** goes further: it handles arrays and dataframes larger than RAM by splitting them into chunks and computing lazily, only materializing results when you call `.compute()`.

In computational biology, single-cell RNA-seq datasets routinely exceed 50 GB. Dask-backed AnnData (the scverse ecosystem) is how those datasets are analyzed without loading everything into memory.

---

## 2. Intuition

**Joblib:** Think of `Parallel(n_jobs=-1)(delayed(f)(x) for x in data)` as a parallel list comprehension. `delayed(f)` wraps `f` so it is not called immediately — instead, it records the call. `Parallel` then dispatches all the recorded calls across workers and collects results.

**Dask array:** A Dask array is a recipe for a NumPy array. It knows what operations to perform, but it has not performed them yet. When you call `.compute()`, Dask executes the recipe in chunks, never loading more than one chunk at a time (by default). The **task graph** is the DAG of operations — Dask can visualize it and optimize it before executing.

---

## 3. Biological background

**Single-cell RNA-seq (scRNA-seq):** Each cell's gene expression is measured as a count matrix — rows are cells (tens of thousands), columns are genes (20,000–30,000). A typical 10x Genomics dataset for 50,000 cells × 30,000 genes has 1.5 billion entries — too large to load as a dense NumPy array (12 GB at float32).

Dask handles this by:
- Chunking along the cell axis (e.g., 5,000 cells per chunk)
- Computing mean/variance per chunk, then combining
- Never holding all 1.5 billion values in RAM simultaneously

**K-mer counting over a metagenome:** A metagenomic sample may have 100 million reads. Parallelizing k-mer counting across reads with joblib is a direct, practical acceleration.

---

## 4. Mathematical explanation

### Joblib memory caching

Joblib `Memory` implements **memoization**: given the same function and same arguments (compared by hash), return the cached result instead of recomputing. For expensive functions called repeatedly (e.g., loading and preprocessing a large FASTA file), this avoids redundant computation across notebook runs.

### Dask chunked mean

For a signal $x$ of length $N$ split into chunks of size $C$:

$$\bar{x} = \frac{1}{N} \sum_{k=0}^{K-1} \sum_{j \in \text{chunk}_k} x_j$$

Dask computes each chunk's partial sum, accumulates, and divides. The result is identical to computing the mean on the full array — it just never requires more than $C$ elements in memory at once.

---

## 5. Computational explanation

### Joblib API
```python
from joblib import Parallel, delayed, Memory

# Parallel map
results = Parallel(n_jobs=-1)(delayed(fn)(x) for x in data)
# n_jobs=-1: use all available CPUs
# n_jobs=4: use 4 CPUs
# n_jobs=-2: use all CPUs minus one

# Caching
cache = Memory(location='/tmp/joblib_cache')
cached_fn = cache.cache(fn)
result = cached_fn(args)  # recomputed first call, cached thereafter
```

### Dask array API
```python
import dask.array as da

x = da.from_array(numpy_array, chunks=(1000,))  # 1000 elements per chunk
y = da.map_blocks(fn, x)   # apply fn to each chunk independently
result = y.compute()        # materialize the result
```

---

## 6. Python implementation

In [ ]:
import numpy as np
import time
import os
import tempfile
import matplotlib.pyplot as plt
from collections import Counter

try:
    from joblib import Parallel, delayed, Memory
    JOBLIB_AVAILABLE = True
    print("joblib available")
except ImportError:
    JOBLIB_AVAILABLE = False
    print("joblib not installed — install with: pip install joblib")

try:
    import dask.array as da
    import dask.dataframe as dd
    import dask
    DASK_AVAILABLE = True
    print("dask available:", dask.__version__)
except ImportError:
    DASK_AVAILABLE = False
    print("dask not installed — install with: pip install dask")

np.random.seed(42)

In [ ]:
# --- (1) Joblib parallel k-mer counting ---

def count_kmers(seq, k=4):
    """Count all k-mers in an integer-encoded sequence.
    Returns a dict {kmer_tuple: count}.
    """
    counts = Counter()
    for i in range(len(seq) - k + 1):
        kmer = tuple(seq[i:i+k])
        counts[kmer] += 1
    return dict(counts)

# Generate 500 sequences
rng = np.random.default_rng(42)
seqs = [rng.integers(0, 4, size=300, dtype=np.int8) for _ in range(500)]

# Serial baseline
t0 = time.perf_counter()
kmer_counts_serial = [count_kmers(s) for s in seqs]
t_serial = time.perf_counter() - t0
print(f"Serial k-mer counting (500 seqs): {t_serial:.4f} s")

# Joblib parallel
if JOBLIB_AVAILABLE:
    t0 = time.perf_counter()
    kmer_counts_parallel = Parallel(n_jobs=-1)(
        delayed(count_kmers)(s) for s in seqs
    )
    t_parallel = time.perf_counter() - t0
    print(f"Joblib parallel (all CPUs): {t_parallel:.4f} s  (speedup: {t_serial/t_parallel:.2f}x)")
    
    # Correctness check
    assert kmer_counts_serial[0] == kmer_counts_parallel[0]
    print("Correctness: PASSED")

In [ ]:
# --- Joblib Memory caching example ---
if JOBLIB_AVAILABLE:
    cache_dir = os.path.join(tempfile.gettempdir(), 'joblib_compbio')
    mem = Memory(location=cache_dir, verbose=0)

    @mem.cache
    def expensive_gc_analysis(seq_array, window_size):
        """Simulate an expensive analysis (e.g., sliding GC over large genome)."""
        time.sleep(0.1)  # simulate slow computation
        cs = np.cumsum((seq_array == 2) | (seq_array == 3))
        cs = np.concatenate(([0], cs))
        return (cs[window_size:] - cs[:-window_size]) / window_size

    test_seq = rng.integers(0, 4, size=10_000, dtype=np.int8)

    t0 = time.perf_counter()
    result1 = expensive_gc_analysis(test_seq, window_size=100)
    t_first = time.perf_counter() - t0

    t0 = time.perf_counter()
    result2 = expensive_gc_analysis(test_seq, window_size=100)
    t_cached = time.perf_counter() - t0

    print(f"First call (computed): {t_first:.3f} s")
    print(f"Second call (cached):  {t_cached:.4f} s  (speedup: {t_first/t_cached:.0f}x)")
    assert np.allclose(result1, result2)
    print("Cache correctness: PASSED")

In [ ]:
# --- (2) Dask array: sliding window GC over a 10M-element signal ---
if DASK_AVAILABLE:
    signal_np = rng.integers(0, 4, size=10_000_000, dtype=np.int8)
    print(f"Signal size: {signal_np.nbytes / 1e6:.1f} MB")
    
    CHUNK = 1_000_000  # 1M elements per chunk
    signal_da = da.from_array(signal_np, chunks=CHUNK)
    print(f"Dask array: {signal_da.shape}, {signal_da.npartitions} chunks")
    
    # Compute mean GC content (simple reduction)
    t0 = time.perf_counter()
    gc_mean_np = float(((signal_np == 2) | (signal_np == 3)).mean())
    t_np = time.perf_counter() - t0
    
    gc_bool_da = (signal_da == 2) | (signal_da == 3)
    t0 = time.perf_counter()
    gc_mean_dask = float(gc_bool_da.mean().compute())
    t_dask = time.perf_counter() - t0
    
    print(f"Mean GC (NumPy, 10M): {gc_mean_np:.4f}  in {t_np:.3f} s")
    print(f"Mean GC (Dask, 10M):  {gc_mean_dask:.4f}  in {t_dask:.3f} s")
    assert abs(gc_mean_np - gc_mean_dask) < 1e-6, "Dask result disagrees!"
    print("Dask correctness: PASSED")
else:
    print("Dask not available — skipping Dask demonstration")
    print("Install with: pip install dask")

In [ ]:
# --- (3) Dask dataframe: groupby on synthetic expression data ---
if DASK_AVAILABLE:
    import pandas as pd
    
    # Synthetic gene expression TSV: 10,000 genes x 6 columns
    n_genes = 10_000
    cell_types = rng.choice(['neuron', 'astrocyte', 'oligodendrocyte', 'microglia'], size=n_genes)
    expr_data = {
        'gene_id': [f'GENE_{i:05d}' for i in range(n_genes)],
        'cell_type': cell_types,
        'sample_1': rng.exponential(scale=5.0, size=n_genes),
        'sample_2': rng.exponential(scale=5.0, size=n_genes),
        'sample_3': rng.exponential(scale=5.0, size=n_genes),
        'sample_4': rng.exponential(scale=5.0, size=n_genes),
    }
    df_pandas = pd.DataFrame(expr_data)
    
    # Dask dataframe from pandas (in practice: from_delayed or read_csv with multiple files)
    ddf = dd.from_pandas(df_pandas, npartitions=4)
    
    # Groupby: mean expression per cell type
    t0 = time.perf_counter()
    mean_expr_dask = ddf.groupby('cell_type')[['sample_1', 'sample_2', 'sample_3', 'sample_4']].mean().compute()
    t_dask_gb = time.perf_counter() - t0
    
    t0 = time.perf_counter()
    mean_expr_pandas = df_pandas.groupby('cell_type')[['sample_1', 'sample_2', 'sample_3', 'sample_4']].mean()
    t_pandas_gb = time.perf_counter() - t0
    
    print("Mean expression by cell type (Dask):")
    print(mean_expr_dask.round(2))
    print(f"\nDask groupby: {t_dask_gb:.4f} s  |  Pandas groupby: {t_pandas_gb:.4f} s")
    print("(Dask overhead visible at small scale; advantage emerges with 100k+ rows/multiple files)")

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Joblib scaling curve
ax = axes[0]
if JOBLIB_AVAILABLE:
    n_jobs_list = [1, 2, 4]
    import multiprocessing
    n_jobs_list = [n for n in n_jobs_list if n <= multiprocessing.cpu_count()]
    joblib_times = []
    for nj in n_jobs_list:
        t0 = time.perf_counter()
        Parallel(n_jobs=nj)(delayed(count_kmers)(s) for s in seqs)
        joblib_times.append(time.perf_counter() - t0)
    ax.plot(n_jobs_list, [t_serial / t for t in joblib_times], 'o-',
            color='#8e44ad', linewidth=2, markersize=8)
    ax.plot(n_jobs_list, n_jobs_list, 'k--', alpha=0.4, label='Ideal linear')
    ax.set_xlabel('n_jobs', fontsize=11)
    ax.set_ylabel('Speedup', fontsize=11)
    ax.set_title('Joblib Parallel K-mer Counting\nSpeedup vs n_jobs', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'joblib not installed', ha='center', va='center',
            transform=ax.transAxes, fontsize=12)
    ax.set_title('Joblib (not installed)', fontweight='bold')

# Panel 2: Dask task graph (text representation)
ax = axes[1]
ax.axis('off')
task_graph_text = (
    "Dask Task Graph (conceptual)\n"
    "for gc_bool_da.mean()\n\n"
    "   signal_da (10M elements)\n"
    "   [chunk 0]  [chunk 1]  ... [chunk 9]\n"
    "       |          |               |\n"
    "   (==2)|(==3)  (==2)|(==3)  ...  \n"
    "       |          |               |\n"
    "   chunk_mean  chunk_mean   chunk_mean\n"
    "       \\         |          /\n"
    "        \\        |         /\n"
    "         weighted_average\n"
    "              |\n"
    "           .compute()\n"
    "              |\n"
    "           result (scalar)"
)
ax.text(0.05, 0.95, task_graph_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#eaf2f8', alpha=0.8))
ax.set_title('Dask Task Graph Structure', fontweight='bold')

# Panel 3: Wall time comparison
ax = axes[2]
labels = ['Serial\nk-mer', 'Joblib\nk-mer']
vals = [t_serial, t_parallel if JOBLIB_AVAILABLE else t_serial]
colors = ['#e74c3c', '#27ae60']
if DASK_AVAILABLE:
    labels += ['NumPy GC\n(10M)', 'Dask GC\n(10M)']
    vals += [t_np, t_dask]
    colors += ['#2980b9', '#8e44ad']
ax.bar(labels, vals, color=colors, edgecolor='black', alpha=0.85)
for i, (lbl, v) in enumerate(zip(labels, vals)):
    ax.text(i, v + 0.002, f'{v:.3f}s', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Wall time (seconds)', fontsize=11)
ax.set_title('Wall Time Comparison', fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../datasets/nb03_joblib_dask.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Exercises

1. **Joblib n_jobs=-2:** Re-run the k-mer counting with `n_jobs=-2` (all CPUs minus 1). How does it differ from `n_jobs=-1`? When would you prefer `-2` over `-1`?

2. **Dask from_delayed:** Create a Dask array from 10 separate NumPy arrays using `da.from_delayed`. Compute the mean across all arrays without concatenating them into memory first.

3. **Memory cache invalidation:** The joblib `Memory` cache key is based on function source code and arguments. Change the `window_size` argument and verify that the cache is recomputed. Then modify the function body (add a comment) — does that invalidate the cache?

4. **Dask groupby on large synthetic dataset:** Generate a DataFrame with 1 million rows and 10 columns. Partition it into 20 Dask partitions. Compute `groupby('category').agg({'val1': 'mean', 'val2': 'std'})`. Compare to pandas on the full dataset.

## 12. Reflection

*Write here: When would you choose Dask over NumPy? When is joblib preferable to `multiprocessing.Pool`? What does "lazy evaluation" mean in the context of Dask, and why is it useful?*

---

## Papers referenced

- Virshup, I. et al. (2024). "The scverse ecosystem for single-cell omics analysis." *Nature Methods* 21:229–232. — Dask-backed AnnData in production single-cell analysis.

## References

- joblib docs: https://joblib.readthedocs.io/en/latest/
- Dask docs: https://docs.dask.org/en/stable/
- Dask best practices: https://docs.dask.org/en/stable/best-practices.html

## Future work / open questions

- Dask distributed scheduler (a separate package) allows computation across multiple machines — the same API, but `client = Client('scheduler-address')` replaces the local scheduler. How would you deploy this on a SLURM cluster? (See NB06.)
- When does joblib's multiprocessing backend under-perform its threading backend? Hint: for very small task payloads, pickling overhead dominates.